# Grid Measure — S300 이동 + B1500 I-V 측정 통합 (초안 v0.2)

희수 선배님 두 코드를 합친 것:
- **① S300 프로버** (`examples/s300/S300_test_v2.ipynb`) : 척(chuck)을 좌표로 이동
- **② B1500A** (`examples/b1500/example_01.ipynb`) : 전압 sweep → 전류 측정

**동작:** 좌표 CSV 를 읽어 각 좌표마다 → `[분리] → [XY 이동] → [접촉] → [B1500 측정] → [CSV 저장]` 반복.

> v0.2: Nucleus4 통신 매뉴얼(`docs/manuals/cascade communication commend.pdf`)로 모든 SCPI 명령 확정. 추측 부분 제거.

## ⚠️ 코드 실행 전 — 사람이 손으로 끝내놔야 하는 준비 (Nucleus UI)
매뉴얼 워크플로우 중 이 단계들은 **코드가 대체하지 않으므로** 먼저 수동으로 완료해야 함:
1. 척 로드 / 진공 ON / 소자 세팅
2. **Alignment** (2-point align) — 안 하면 좌표가 통째로 어긋남
3. **Tipping / Set Contact** — 팁 contact 높이를 잡아둠
4. **첫 소자에 팁을 직접 contact** 시킨 상태로 둠 → 아래 5번 셀에서 그 위치를 원점·contact높이로 등록

> 측정 중 light off 등은 Nucleus 쪽 물리 작업이라 코드 밖.

In [1]:
import os
import time
import pandas as pd
import pyvisa
from pymeasure.instruments.agilent import AgilentB1500

# --- pandas 3.0 호환 shim -----------------------------------------------------
# pandas 2.1 에서 DataFrame.applymap -> DataFrame.map 으로 이름 바뀌고
# pandas 3.0 에서 applymap 이 제거됨. 그런데 pymeasure 0.16 의 B1500 read_data 가
# 아직 applymap 을 써서 측정 데이터 읽을 때 에러남. map 이 applymap 과 동작 동일하므로
# 옛 이름을 다시 연결해 호환을 살린다. (라이브러리가 pandas 3 지원하면 삭제 가능)
if not hasattr(pd.DataFrame, "applymap"):
    pd.DataFrame.applymap = pd.DataFrame.map

## 0. 설정 (여기만 바꾸면 됨)

In [2]:
# --- 장비 주소 ---
S300_GPIB  = "GPIB0::28::INSTR"
B1500_GPIB = "GPIB0::17::INSTR"

# --- 좌표 CSV (형식: Subsite Name, X Position, Y Position, Note), 단위 = micron ---
# 테스트용(3점, 최대 100um 이동). 실제 측정 시 아래 old 경로로 되돌릴 것.
COORD_CSV = "../../utils/test_coordinates.csv"
# COORD_CSV = "../../utils/old/grid_coordinates.csv"   # 실제 측정용 (160점)

# --- 드라이브할 전압 범위 (drive="sweep" 인 SMU 에 적용) ---
V_START      = 0.0     # 시작 전압 [V]
V_STOP       = 1.0     # 끝 전압 [V]
V_POINTS     = 11      # sweep 포인트 수
I_COMPLIANCE = 1e-3    # 전류 컴플라이언스 [A]

# --- 팁(프로브) ↔ SMU ↔ 역할 매핑 -------------------------------------------
# 팁 4개가 각각 어느 단자(Gate/Drain/Source/Bulk)에 닿는지는 셋업마다 다르므로
# 여기서 직접 지정한다.  키 N = B1500 채널 번호 smuN  (UNT? 로 장착 확인).
#   role    : 표기용 이름 (자유)
#   drive   : "sweep" → V_START~V_STOP sweep (보통 Gate)
#             "const" → const_v 로 DC 고정 (보통 Drain/Source/Bulk)
#             "off"   → 사용 안 함(disable)
#   const_v : drive="const" 일 때 인가 전압 [V]
SMU_CONFIG = {
    1: {"role": "GATE",   "drive": "sweep", "const_v": None},
    2: {"role": "DRAIN",  "drive": "const", "const_v": 1.0},
    3: {"role": "SOURCE", "drive": "const", "const_v": 0.0},
    4: {"role": "BULK",   "drive": "off",   "const_v": 0.0},
}

# --- S300 척(chuck) Device ID ---
# 매뉴얼 p.37: 2 = Elite/Summit/S300/Alessi 의 척 제어 device ID (축 번호 아님)
CHUCK_ID = 2

# --- 결과 저장 폴더 ---
OUT_DIR = "results"

## 1. S300 프로버 제어 (① 코드)
`S300_test_v2.ipynb` 의 `CascadeS300` 에 매뉴얼 기반 편의 함수 추가.

**기준점/안전 동작 (매뉴얼 근거)**
- `set_reference()` : 사람이 첫 소자에 contact 시킨 현재 위치를 등록 — `:set:pres 2 0 0`(현재 XY=원점, p.154 "Set Zero") + `:set:cont 2 <z>`(현재 Z=contact 높이, p.139).
- `move_xy()` : `:mov:abs 2 X Y none` — **Z를 안전높이로 자동 분리한 뒤 XY 이동**(p.56)이라 이동 중 소자가 안 긁힌다.
- `contact()`/`separate()` : `:mov:cont`/`:mov:sep` — 사람이 set한 contact 높이로 복귀/분리. 그 아래로는 안 내려가 소자를 찍지 않는다.

In [3]:
class CascadeS300:
    def __init__(self, gpib_address=S300_GPIB):
        self.rm = pyvisa.ResourceManager()
        self.instrument = self.rm.open_resource(gpib_address)
        self.instrument.timeout = 30000      # 30 sec: 이동 완료(COMPLETE)까지 대기 (매뉴얼 권장)

    def _write(self, command):
        """메타 명령($:...) 등 '응답 없는' 명령 전용 — write 만."""
        self.instrument.write(command)

    def ask(self, command):
        """질의(? 명령) 또는 resp-on 상태의 명령. 응답 문자열 반환.
        $:set:resp on 상태에서 → 명령은 'COMPLETE', 질의는 값, 실패는 '@에러' 를 돌려줌."""
        try:
            return self.instrument.query(command).strip()
        except Exception as e:
            return f"Error: {e}"

    def cmd(self, command):
        """action/설정 명령 실행. $:set:resp on 덕에 완료되면 'COMPLETE' 반환(=완료까지 대기).
        응답이 '@' 로 시작하면 프로버 에러 → 예외 발생."""
        resp = self.ask(command)
        if resp.startswith("@"):
            raise RuntimeError(f"S300 명령 실패: {command!r} -> {resp!r}")
        return resp

    def setup(self):
        """매뉴얼 정식 원격 셋업 (Nucleus4 가이드 p.9). 순서 중요:
        $:set:resp on 을 먼저 켜야 이후 이동/설정 명령이 'COMPLETE' 응답을 줘서
        타임아웃 없이 완료를 확인할 수 있다. (이게 빠지면 모든 action 명령이 타임아웃)
        ※ 이 프로버 펌웨어는 :set: 명령에 device ID(CHUCK_ID)를 요구함(:set:unit 2 metric)."""
        self._write("$:set:mode summit")             # 명령 해석 모드 (meta, 응답없음)
        self._write("$:set:resp on")                 # ★ 명령마다 COMPLETE/에러 응답 켜기 (meta, 응답없음)
        self.cmd(":SYST:OPER:MODE REMOTE")           # 원격 모드 (device ID 불필요)
        self.cmd(f":set:unit {CHUCK_ID} metric")     # 단위 = micron (device ID 필요)
        return "OK (mode=summit, resp=on, REMOTE, metric)"

    # 하위호환 (개별 호출용)
    def set_remote(self):
        return self.cmd(":SYST:OPER:MODE REMOTE")

    def set_metric(self):
        return self.cmd(f":set:unit {CHUCK_ID} metric")   # device ID 필요

    # --- 위치 ---
    def read_position(self):
        """현재 척 좌표 (x, y, z) micron 반환 (:mov:abs? , p.62)."""
        resp = self.ask(f":mov:abs? {CHUCK_ID}")
        x, y, z = (float(v) for v in resp.replace(",", " ").split()[:3])
        return x, y, z

    def set_reference(self):
        """사람이 첫 소자에 contact 시킨 현재 위치를 원점(0,0)+contact높이로 등록.
        - 현재 XY → 원점 (:set:pres, p.154)   /   현재 Z → contact 높이 (:set:cont, p.139)"""
        x, y, z = self.read_position()
        self.cmd(f":set:pres {CHUCK_ID} 0 0")        # 현재 XY = 원점
        self.cmd(f":set:cont {CHUCK_ID} {int(z)}")   # 현재 Z = contact 높이
        print(f"기준점 등록: 현재위치 {(x, y, z)} → 원점(0,0), contact Z={int(z)}")

    # --- 분리 / 접촉 / 이동 (COMPLETE = 동작 완료. 별도 busy 폴링 불필요) ---
    def separate(self):
        """팁 분리 (:mov:sep, p.89). 완료(COMPLETE)까지 대기."""
        return self.cmd(f":mov:sep {CHUCK_ID}")

    def contact(self):
        """팁 접촉 — set된 contact 높이로 (:mov:cont, p.63). 완료까지 대기."""
        return self.cmd(f":mov:cont {CHUCK_ID}")

    def move_xy(self, dx, dy):
        """원점 기준 (dx,dy) micron 으로 XY 이동 (:mov:abs, z=none → Z 안전높이 자동분리).
        완료(COMPLETE)까지 대기 후 실제 위치 (x,y,z) 반환."""
        self.cmd(f":mov:abs {CHUCK_ID} {dx} {dy} none")
        return self.read_position()

## 2. B1500 측정 (② 코드)
`example_01.ipynb` 의 staircase sweep 을 함수로 묶고, **`SMU_CONFIG` 로 팁별 역할(sweep/const/off)을 선택**할 수 있게 함. `LINEAR_DOUBLE` = 왕복 sweep 이라 포인트는 `2*nop`.

In [4]:
def drain_b1500_errors(b1500, max_reads=500):
    """B1500 FLEX 에러큐를 읽어서 끝까지 비운다(+0 'No Error' 나올 때까지).
    *CLS 로는 이 큐가 안 비워질 수 있어, 이전 통신오류(NCIC 등)로 쌓인
    +100 backlog 를 직접 제거한다. 비운 에러 개수를 반환."""
    n = 0
    for _ in range(max_reads):
        resp = b1500.ask("ERRX?")
        if resp.split(",")[0].strip() in ("0", "+0"):
            return n
        n += 1
    return n  # max_reads 까지 못 비우면 그대로 반환(능동 생성 의심)


def setup_b1500():
    """B1500 연결 및 초기화"""
    # 참고: 최신 pymeasure(0.16+)의 AgilentB1500 는 read/write_termination 을
    # 내부에서 "\r\n" 으로 자동 설정하므로 여기서 넘기면 안 됨(중복 인자 에러).
    b1500 = AgilentB1500(
        B1500_GPIB,
        timeout=600000,
    )
    # 연결 직후 청소: 이전 세션/통신오류(NCIC 등)로 남은 입력버퍼·에러큐를 비운다.
    b1500.clear()                      # GPIB device clear (입력버퍼/상태 리셋)
    b1500.write("*CLS")                # 표준 상태 클리어
    n = drain_b1500_errors(b1500)      # FLEX 에러큐 직접 비우기 (*CLS 로 안 비워지는 +100 backlog)
    if n:
        print(f"[setup] B1500 에러큐에서 묵은 에러 {n}개 제거함")

    b1500.initialize_all_smus()
    b1500.data_format(21, mode=1)   # SMU 초기화 후 호출
    return b1500


def measure_iv(b1500, v_start, v_stop, nop, compliance, config):
    """config(SMU_CONFIG) 에 따라 각 SMU(=팁)를 sweep/const 로 설정하고 I-V 측정.
    반환: DataFrame (포인트 2*nop, LINEAR_DOUBLE 왕복 sweep)."""
    sweep_chs = [ch for ch, c in config.items() if c["drive"] == "sweep"]
    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = sweep_chs + const_chs
    if not sweep_chs:
        raise ValueError("sweep 할 SMU 가 없습니다. SMU_CONFIG 에서 하나는 drive='sweep' 이어야 함.")

    smus = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    b1500.meas_mode("STAIRCASE_SWEEP", *[smus[ch] for ch in active])
    for ch in active:
        s = smus[ch]
        s.enable()
        s.adc_type = "HRADC"
        s.meas_range_current = "1 uA"
        s.meas_op_mode = "COMPLIANCE_SIDE"

    b1500.adc_setup("HRADC", "AUTO", 6)
    b1500.sweep_timing(0, 0.5, step_delay=0.1)        # hold, delay
    b1500.sweep_auto_abort(False, post="STOP")

    # 메인 sweep SMU (Gate 등)
    main_ch = sweep_chs[0]
    smus[main_ch].staircase_sweep_source(
        "VOLTAGE", "LINEAR_DOUBLE", "Auto Ranging",
        v_start, v_stop, nop, compliance,
    )
    # 추가로 sweep 하는 SMU 가 있으면 동기 sweep
    for ch in sweep_chs[1:]:
        smus[ch].synchronous_sweep_source("VOLTAGE", "Auto Ranging", v_start, v_stop, compliance)
    # const SMU (Drain/Source/Bulk 등) 는 DC 고정
    for ch in const_chs:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", config[ch]["const_v"], stepsize=0.1, pause=20e-3)

    # 측정 시작
    b1500.check_errors()
    b1500.clear_buffer()
    b1500.clear_timer()
    b1500.send_trigger()

    # 끝날 때까지 대기 후 한 번에 읽기
    b1500.check_idle()
    data = b1500.read_data(2 * nop)

    # const SMU 0V 로 복귀
    for ch in const_chs:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, stepsize=0.1, pause=20e-3)
    return data

## 2.5 (선택) Sampling 측정 — 시간에 따른 전류 (transient / 안정성)

`measure_iv` 는 전압을 **쓸면서**(sweep) I-V 곡선을 얻고, `measure_sampling` 은 전압을 **고정**한 채 일정 시간 간격마다 전류를 찍어 **시간축(I-t) 데이터**를 얻습니다. (B1500 매뉴얼의 `SAMPLING` 모드)

- 같은 `SMU_CONFIG` 를 그대로 재사용 — `sweep` 역할 SMU(GATE)는 `V_STOP`(또는 `SAMP_SWEEP_BIAS`)로 고정, `const` 역할은 `const_v` 로 고정, `off` 는 미사용.
- 반환 `DataFrame` 에는 **time stamp 열 + 채널별 전류 열**이 들어 있어 `전류 vs 시간` 으로 바로 그릴 수 있음.
- 용도: bias-stress 안정성, 접촉 안정화 확인, drift 모니터링 등. **필요 없으면 호출 안 하면 됨** (I-V 측정엔 영향 없음).

> ⚠️ `measure_iv`(STAIRCASE_SWEEP) 와 `measure_sampling`(SAMPLING) 은 측정 모드를 서로 바꾸므로, 한 좌표에서 **둘 다** 쓸 경우 순서대로(예: I-V 먼저 → sampling 나중) 호출하면 됩니다. 각자 자기 모드를 다시 설정하므로 충돌 없음.

In [ ]:
# --- Sampling(시간) 측정 파라미터 (필요할 때만) -------------------------------
# I-V(sweep)와 달리 전압을 '고정'해두고 일정 시간 간격마다 전류를 찍어
# 시간에 따른 변화(transient / bias-stress / 안정성)를 본다.
SAMP_INTERVAL  = 0.01    # 샘플 간격 [s]  (>=0.002 권장, 그 미만은 고속/제약 있음)
SAMP_NUMBER    = 200     # 샘플 개수  (총 측정시간 ≈ INTERVAL * NUMBER)
SAMP_HOLD_BIAS = 0       # bias 인가 후 첫 측정까지 대기시간 [s]
SAMP_BASE_V    = 0.0     # base(측정 전/후) 전압 [V]

# sweep 역할 SMU(예: GATE)를 sampling 중엔 어떤 DC 전압으로 고정할지.
# None 이면 V_STOP('on' 전압)을 사용. const 역할은 SMU_CONFIG 의 const_v 그대로.
SAMP_SWEEP_BIAS = None   # 예: 1.0 으로 두면 게이트를 1V 고정한 채 시간 측정


def measure_sampling(b1500, config, interval=SAMP_INTERVAL, number=SAMP_NUMBER,
                     hold_bias=SAMP_HOLD_BIAS, base=SAMP_BASE_V,
                     sweep_bias=SAMP_SWEEP_BIAS, compliance=I_COMPLIANCE):
    """SAMPLING 모드: 각 SMU(=팁)에 DC bias 를 걸고 '시간에 따른 전류'를 측정.
    - sweep 역할 SMU(GATE 등) → sweep_bias(None 이면 V_STOP)로 고정
    - const 역할 SMU         → SMU_CONFIG 의 const_v 로 고정
    - off                    → 사용 안 함
    반환: DataFrame (number 포인트, time stamp + 채널별 전류).
    measure_iv 와 같은 SMU_CONFIG 를 그대로 재사용한다."""
    sweep_chs = [ch for ch, c in config.items() if c["drive"] == "sweep"]
    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = sweep_chs + const_chs
    if not active:
        raise ValueError("sampling 할 SMU 가 없습니다. SMU_CONFIG 확인.")

    smus = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    # 측정 모드 = SAMPLING (측정 순서 = active 순서)
    b1500.meas_mode("SAMPLING", *[smus[ch] for ch in active])
    for ch in active:
        s = smus[ch]
        s.enable()
        s.adc_type = "HSADC"            # sampling 은 고속 ADC 사용
        s.meas_range_current = "1 uA"
        s.meas_op_mode = "COMPLIANCE_SIDE"

    b1500.sampling_mode = "LINEAR"
    b1500.adc_setup("HSADC", "AUTO", 1)
    b1500.sampling_timing(hold_bias, interval, number)   # MT: hold, interval, #points
    b1500.sampling_auto_abort(False, post="Bias")        # 중도중단 off
    b1500.time_stamp = True                              # 시간축 기록 ON

    # 각 SMU 에 DC bias 인가 (base → bias)
    for ch in active:
        bias = (sweep_bias if sweep_bias is not None else V_STOP) if ch in sweep_chs \
            else config[ch]["const_v"]
        smus[ch].sampling_source("VOLTAGE", "Auto Ranging", base, bias, compliance)

    # 측정 시작
    b1500.check_errors()
    b1500.clear_buffer()
    b1500.clear_timer()
    b1500.send_trigger()

    # 끝날 때까지 대기 후 한 번에 읽기 (time stamp + current 포함)
    b1500.check_idle()
    data = b1500.read_data(number)

    # 모든 SMU 0V 로 복귀
    for ch in active:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, stepsize=0.1, pause=20e-3)
    return data

## 3. 장비 연결

오류가 자주 나는데, vscode를 껐다 키고, B1500을 껐다 키고, 잭 연결이 잘 되어있는지 확인해보기

In [5]:
s300 = CascadeS300(S300_GPIB)
print("S300:", s300.ask("*IDN?"))
print("S300 setup:", s300.setup())   # $:set:mode summit + $:set:resp on + REMOTE + metric (순서 중요)

b1500 = setup_b1500()
print("B1500:", b1500.ask("*IDN?"))

S300: Cascade Microtech, S300 Theta, 586480504, 3, 3
S300 setup: OK (mode=summit, resp=on, REMOTE, metric)
[setup] B1500 에러큐에서 묵은 에러 2개 제거함
B1500: Agilent Technologies,B1500A,0,A.06.02.2023.0401


## 3.5 통신 확인 (S300 ↔ B1500)
좌표 이동(S300)·측정(B1500) 두 장비가 모두 응답하는지, S300 이 REMOTE/정렬 완료 상태인지 먼저 확인합니다.
**[OK] 가 떠야 다음으로 진행하세요.** (매뉴얼 p.9 `Verifying GPIB Communication` 절에 해당)

In [6]:
def check_comm(s300, b1500):
    """S300(좌표이동) + B1500(측정) 통신/준비상태 확인."""
    print("=== 통신 확인 ===")
    ok = True

    # --- S300 ---
    idn = s300.ask("*IDN?")
    print("S300 *IDN?        :", idn)
    ok &= "Cascade" in idn

    mode = s300.ask("$:set:mode?")          # SUMMIT/EG = Nucleus 인터프리터 동작중
    print("S300 interpreter  :", mode)
    ok &= mode in ("SUMMIT", "EG")

    tst = s300.ask("*tst?")                 # 0 = self-test 정상
    print("S300 self-test    :", tst)
    ok &= tst.strip() == "0"

    s300.set_remote()
    oper = s300.ask(":SYST:OPER:MODE?")     # REMOTE 여야 원격제어 가능
    print("S300 oper mode    :", oper)
    ok &= oper == "REMOTE"

    align = s300.ask(":align:wafer:busy?")  # SUCCESS = 수동 alignment 끝난 상태인지 확인용
    print("S300 align status :", align)     # (정렬 안 됐으면 좌표 이동이 어긋남)

    # --- B1500 ---
    bidn = b1500.ask("*IDN?")
    print("B1500 *IDN?       :", bidn)
    ok &= "B1500" in bidn

    opc = b1500.ask("*OPC?")                # 1 = idle/완료
    print("B1500 *OPC?       :", opc)
    ok &= opc.strip() == "1"

    print("B1500 UNT?        :", b1500.ask("UNT?"))   # 장착 모듈 목록

    print("\n결과:", "[OK] 통신 정상" if ok else "[FAIL] 확인 필요 - 위 항목 점검")
    return ok


check_comm(s300, b1500)

=== 통신 확인 ===
S300 *IDN?        : Cascade Microtech, S300 Theta, 586480504, 3, 3
S300 interpreter  : SUMMIT
S300 self-test    : 0
S300 oper mode    : REMOTE
S300 align status : SUCCESS
B1500 *IDN?       : Agilent Technologies,B1500A,0,A.06.02.2023.0401
B1500 *OPC?       : 1
B1500 UNT?        : B1530A,0;B1517A,0;B1517A,0;B1517A,0;0,0;0,0;0,0;0,0;0,0;0,0

결과: [OK] 통신 정상


True

## 4. 좌표 파일 읽기 (CSV / 엑셀 자동 판별)
`COORD_CSV` 의 확장자가 `.csv` 면 `read_csv`, `.xlsx/.xls` 면 `read_excel` 로 읽습니다. 둘 다 `Subsite Name, X Position, Y Position` 컬럼 구조는 동일.

In [7]:
# 확장자 보고 CSV / 엑셀 자동 판별 (.xlsx 는 openpyxl 필요: pip install openpyxl)
ext = os.path.splitext(COORD_CSV)[1].lower()
if ext in (".xlsx", ".xls"):
    coords = pd.read_excel(COORD_CSV)
else:
    coords = pd.read_csv(COORD_CSV)

print(f"좌표 {len(coords)} 개 로드 완료: {COORD_CSV}")
coords.head()

좌표 3 개 로드 완료: ../../utils/test_coordinates.csv


,Subsite Name,X Position,Y Position,Note
0,1,0,0,origin (사람이 contact시킨 자리 - 이동없음)
1,2,100,0,오른쪽 100um
2,3,0,100,위 100um


## 5. 기준점 등록 — ⚠️ 사람이 첫 소자에 팁 contact 시킨 상태에서 실행
위 '준비' 단계대로 **사람이 첫 소자에 팁을 직접 contact** 시킨 상태에서 아래 셀을 실행하세요.
현재 위치가 모든 좌표의 원점(0,0)·contact 높이로 등록됩니다 (실제 contact 위치 기준이라 이후 소자를 찍지 않음).

In [8]:
s300.set_reference()

기준점 등록: 현재위치 (0.0, 0.0, 1.0) → 원점(0,0), contact Z=1


## 6. 메인 루프 — 좌표대로 쭉 이동 + 측정 (① + ② 합치는 곳)

**좌표 convention**: 파일의 X/Y 는 **원점(사람이 contact시킨 소자=0,0) 기준 상대좌표**이고 음수도 가능 (예: 좌상단 `-100, 100`). 보통은 처음 찍는 소자를 `0,0` 으로 둔다.

- **좌표 (0,0)** 인 줄 = 원점 = 사람이 이미 contact → 이동 없이 바로 측정
- **그 외** = `separate()`(분리) → `move_xy()`(XY 이동, Z 자동 분리) → `contact()`(접촉) → 측정

> 💡 처음엔 측정 없이 **이동만** 확인하려면 아래 `measure_iv(...)` 와 저장 줄을 주석 처리하세요.

In [28]:
os.makedirs(OUT_DIR, exist_ok=True)

# 측정 모드 선택: "iv"=I-V만 | "sampling"=시간측정만 | "both"=둘 다
MEASURE_MODE = "iv"

assert MEASURE_MODE in ("iv", "sampling", "both"), 'MEASURE_MODE 는 "iv"/"sampling"/"both" 중 하나'
do_iv   = MEASURE_MODE in ("iv", "both")
do_samp = MEASURE_MODE in ("sampling", "both")
print(f"측정 모드: {MEASURE_MODE}  (I-V={do_iv}, sampling={do_samp})")

for _, row in coords.iterrows():
    sub_id = row["Subsite Name"]
    dx = int(row["X Position"])   # 원점(사람이 contact시킨 소자) 기준 상대좌표 (음수 가능)
    dy = int(row["Y Position"])

    if dx == 0 and dy == 0:
        # 좌표 (0,0) = 원점 = 사람이 이미 contact 시킨 소자 → 이동 없이 바로 측정
        print(f"[Subsite {sub_id}] 원점 (0, 0) — 이동 없음")
    else:
        print(f"[Subsite {sub_id}] -> 원점+({dx}, {dy}) 이동")
        s300.separate()         # 팁 분리
        s300.move_xy(dx, dy)    # XY 이동 (Z 자동 분리 후 이동)
        s300.contact()          # 접촉 (set된 contact 높이로)
        time.sleep(0.2)         # 콘택트 안정화 대기

    # ① I-V 측정 (STAIRCASE_SWEEP) — 팁 역할은 SMU_CONFIG 따름
    if do_iv:
        data = measure_iv(b1500, V_START, V_STOP, V_POINTS, I_COMPLIANCE, SMU_CONFIG)
        out_path = os.path.join(OUT_DIR, f"subsite_{sub_id}.csv")
        data.to_csv(out_path)
        print(f"  저장(I-V): {out_path}")

    # ② sampling 측정 (SAMPLING, 시간축) — 같은 SMU_CONFIG 재사용, 별도 파일
    if do_samp:
        samp = measure_sampling(b1500, SMU_CONFIG)
        samp_path = os.path.join(OUT_DIR, f"subsite_{sub_id}_sampling.csv")
        samp.to_csv(samp_path)
        print(f"  저장(sampling): {samp_path}")

측정 모드: iv  (I-V=True, sampling=False)
[Subsite 1] 원점 (0, 0) — 이동 없음
  저장(I-V): results\subsite_1.csv
[Subsite 2] -> 원점+(100, 0) 이동


RuntimeError: S300 명령 실패: ':mov:sep 2' -> '@Error moving chuck to separate position\n":MOVe:SEParate"'

## 7. 마무리 — 팁 분리 후 원점 복귀

In [ ]:
s300.separate()      # 팁 분리
s300.move_xy(0, 0)   # 원점으로 복귀 (Z 자동 분리 상태로 이동)
print("측정 완료.")

In [36]:
# --- 종료: S300 LOCAL 복귀 + 세션 정리 -------------------------------------
# 케이블 뽑기/세션 끝내기 전에 실행. (척을 움직이지 않는 안전한 명령)
print("S300 -> LOCAL:", s300.ask(":SYST:OPER:MODE LOCAL"))   # 원격모드 해제 (UI 수동조작 복귀)

# VISA 세션 닫기 (생략해도 커널 Restart 시 자동 해제됨)
try:
    s300.instrument.close()
    b1500.adapter.close()
    print("VISA 세션 닫음.")
except Exception as e:
    print("close 중 예외(무시 가능):", e)


S300 -> LOCAL: COMPLETE
VISA 세션 닫음.


In [15]:
import time
D = 10000   # 이동 거리 [µm] = 1cm  (더 키우려면 이 숫자만 바꾸기)
print("시작:", s300.read_position())
for target in [(D, 0), (D, D), (0, D), (0, 0)]:
    print(f"→ 이동 {target}:", s300.move_xy(*target))
    time.sleep(2)   # 각 모서리 2초 멈춤 — 눈으로 확인


시작: (0.0, 0.0, 1.0)
→ 이동 (10000, 0): (10000.0, 0.0, 1.0)
→ 이동 (10000, 10000): (10000.0, 10000.0, 1.0)
→ 이동 (0, 10000): (1.0, 10000.0, 1.0)
→ 이동 (0, 0): (0.0, 0.0, 1.0)


In [9]:
import time, os
os.makedirs(OUT_DIR, exist_ok=True)

D = 10000   # 이동 거리 [µm] = 1cm  (숫자만 바꾸면 조절)
targets = [(0, 0), (D, 0), (D, D), (0, D), (0, 0)]   # 원점→오른쪽→위→왼쪽→원점

print("시작 위치:", s300.read_position())
for i, (dx, dy) in enumerate(targets):
    pos = s300.move_xy(dx, dy)                 # ① XY 이동 (팁 비접촉, Z는 안전높이)
    print(f"[{i}] 이동 {(dx, dy)} → 실제위치 {pos}")

    data = measure_iv(b1500, V_START, V_STOP, V_POINTS, I_COMPLIANCE, SMU_CONFIG)  # ② 측정 (개방→≈0)
    path = os.path.join(OUT_DIR, f"movemeas_{i}_{dx}_{dy}.csv")
    data.to_csv(path)
    print(f"     측정 저장: {path}  (전류≈0 예상, {len(data)}포인트)")

    time.sleep(1)   # 눈으로 확인용 잠깐 멈춤
print("완료")


시작 위치: (0.0, 0.0, 1.0)
[0] 이동 (0, 0) → 실제위치 (0.0, 0.0, 1.0)
     측정 저장: results\movemeas_0_0_0.csv  (전류≈0 예상, 22포인트)
[1] 이동 (10000, 0) → 실제위치 (10000.0, 0.0, 1.0)
     측정 저장: results\movemeas_1_10000_0.csv  (전류≈0 예상, 22포인트)
[2] 이동 (10000, 10000) → 실제위치 (10000.0, 10000.0, 1.0)
     측정 저장: results\movemeas_2_10000_10000.csv  (전류≈0 예상, 22포인트)
[3] 이동 (0, 10000) → 실제위치 (0.0, 10000.0, 1.0)
     측정 저장: results\movemeas_3_0_10000.csv  (전류≈0 예상, 22포인트)
[4] 이동 (0, 0) → 실제위치 (0.0, 0.0, 1.0)
     측정 저장: results\movemeas_4_0_0.csv  (전류≈0 예상, 22포인트)
완료
